## 准备数据

In [16]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [17]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [18]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        # 使用 tf.Variable 声明可训练的变量，并用正态分布初始化权重
        self.W1 = tf.Variable(tf.random.normal(shape=[28 * 28, 100], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros(shape=[100]))
        
        self.W2 = tf.Variable(tf.random.normal(shape=[100, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros(shape=[10]))
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        # 1. 展平图像: (N, 28, 28) -> (N, 784)
        x_flat = tf.reshape(x, [-1, 28 * 28])
        
        # 2. 隐藏层 + ReLU 激活函数
        h1 = tf.matmul(x_flat, self.W1) + self.b1
        a1 = tf.nn.relu(h1)
        
        # 3. 输出层计算 logits (不需要接 softmax，因为 loss 函数自带了)
        logits = tf.matmul(a1, self.W2) + self.b2
        ####################
        return logits
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [ ]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    optimizer.apply_gradients(zip(grads, trainable_vars)) # 使用强大的 Adam 优化器自动更新参数

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [ ]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer,
                                    tf.constant(train_data[0], dtype=tf.float32),
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model,
                    tf.constant(test_data[0], dtype=tf.float32),
                    tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.439318 ; accuracy 0.13276666
epoch 1 : loss 2.3263583 ; accuracy 0.1691
epoch 2 : loss 2.2256417 ; accuracy 0.21376666
epoch 3 : loss 2.1340256 ; accuracy 0.26831666
epoch 4 : loss 2.0487516 ; accuracy 0.33073333
epoch 5 : loss 1.9676961 ; accuracy 0.39315
epoch 6 : loss 1.8893864 ; accuracy 0.4525
epoch 7 : loss 1.8128899 ; accuracy 0.50405
epoch 8 : loss 1.7377659 ; accuracy 0.55005
epoch 9 : loss 1.6639003 ; accuracy 0.5893667
epoch 10 : loss 1.5913268 ; accuracy 0.6239
epoch 11 : loss 1.5202878 ; accuracy 0.6541833
epoch 12 : loss 1.4510798 ; accuracy 0.6807333
epoch 13 : loss 1.3839979 ; accuracy 0.7022833
epoch 14 : loss 1.3192607 ; accuracy 0.7191333
epoch 15 : loss 1.2570423 ; accuracy 0.73425
epoch 16 : loss 1.1974909 ; accuracy 0.74663335
epoch 17 : loss 1.1407105 ; accuracy 0.75783336
epoch 18 : loss 1.0868239 ; accuracy 0.7675
epoch 19 : loss 1.0359113 ; accuracy 0.7759167
epoch 20 : loss 0.98812544 ; accuracy 0.78353333
epoch 21 : loss 0.9435324 ; accuracy